In [ ]:
import datetime

import folium
from matplotlib import pyplot as plt
import numpy as np

from theia.config import SIDC
from theia.coordinates import CoordinateTransformations
from theia.distance import line_of_sight_distance
from theia.terrain import SrtmTerrainModel
from theia.types import ConstantRcsModel, Point, Trajectory

# Introduction

This notebook models a hypothetical attack from Kaliningrad to Bern.

Assumptions:

- All forces but gravity are neglected.
- The motion starts at a fixed initial velocity.

# Parameters

In [ ]:
terrain = SrtmTerrainModel()

start_lat = 54.7640
start_lon = 20.4080
start_alt = 30.0
p_start = Point(
    lat=start_lat,
    lon=start_lon,
    alt=start_alt,
)

stop_lat = 46.946667
stop_lon = 7.444167
stop_alt = terrain.elevationAt(46.946667, 7.444167)
p_stop = Point(
    lat=stop_lat,
    lon=stop_lon,
    alt=stop_alt,
)

# Simulation code

In [ ]:
def build_trajectory(alpha: float, p_start: Point, p_stop: Point):
    """
    Calculate a ballistic trajectory that connects the start and stop point.

    The only force considered is gravity.

    Parameters
    ----------
    alpha: float
        Start angle of the ballistic motion [rad] in the 2D coordinate system
        (see "Returns")
    p_start: Point
        Start point of the trajectory
    p_stop: Point
        Stop point of the trajectory

    Returns
    -------
    v0: float
        Initial velocity [m/s]
    times: np.ndarray
        Time steps of the trajectory [s]. Shape (N,).
    trajectory: np.ndarray
        Points along the trajectory at 10s resolution. Shape (N, 2).
        Column 0 contains the horizontal component, column 1 the vertical component
        of the trajectory.
        The coordinate system is spanned by the LOS from start to end point and
        the parto of radial direction at the start point that is orthogonal to the LOS.
    """
    distance = line_of_sight_distance(*p_start.as_tuple(), *p_stop.as_tuple())

    p_start_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_start.as_tuple())
    )
    p_stop_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_stop.as_tuple())
    )
    direction = p_stop_ecef - p_start_ecef
    direction /= np.linalg.norm(direction)

    v0 = np.sqrt(9.81 * distance / (2 * np.cos(alpha) * np.sin(alpha)))
    t_stop = 2 * np.sin(alpha) * v0 / 9.81

    times = np.arange(0, t_stop, 10).tolist() + [t_stop]
    xs = []
    ys = []
    for t in times:
        x = np.cos(alpha) * v0 * t
        y = np.sin(alpha) * v0 * t - 0.5 * 9.81 * t**2
        xs.append(x)
        ys.append(y)

    return v0, times, np.vstack([xs, ys]).T


def build_earth_curvature(p_start, p_stop, altitude=0):
    distance = line_of_sight_distance(*p_start.as_tuple(), *p_stop.as_tuple())

    p_start_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_start.as_tuple())
    )
    p_stop_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_stop.as_tuple())
    )
    direction = p_stop_ecef - p_start_ecef
    direction /= np.linalg.norm(direction)

    xs = list(np.arange(0, distance, 100))
    ys = []
    for d in xs:
        p_los_ecef = p_start_ecef + d * direction
        p_los_geodetic = CoordinateTransformations.cartesian_to_geodetic(*p_los_ecef)
        y = altitude - p_los_geodetic[2]
        ys.append(y)

    return np.vstack([xs, ys]).T


def convert_to_geodetic(
    p_start: Point,
    p_stop: Point,
    trajectory: np.ndarray,
) -> list[Point]:
    p_start_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_start.as_tuple())
    )
    p_stop_ecef = np.array(
        CoordinateTransformations.geodetic_to_cartesian(*p_stop.as_tuple())
    )

    x_direction = p_stop_ecef - p_start_ecef
    x_direction /= np.linalg.norm(x_direction)
    y_direction = p_start_ecef / np.linalg.norm(p_start_ecef)
    y_direction -= np.dot(x_direction, y_direction) * x_direction
    assert np.isclose(np.dot(x_direction, y_direction), 0)

    points: list[Point] = []
    for x, y in trajectory:
        p_ecef = p_start_ecef + x * x_direction + y * y_direction
        p = CoordinateTransformations.cartesian_to_geodetic(*p_ecef)
        points.append(Point(lat=p[0], lon=p[1], alt=p[2]))
    return points

# Simulation

In [ ]:
alpha = 45.0
v0, times, trajectory_data = build_trajectory(np.deg2rad(alpha), p_start, p_stop)

In [ ]:
points = convert_to_geodetic(p_start, p_stop, trajectory_data)
trajectory = Trajectory(
    target_id=100,
    target_sidc=SIDC.RED_MISSILE,
    times=[datetime.datetime.fromtimestamp(t, tz=datetime.UTC) for t in times],
    lats=[p.lat for p in points],
    lons=[p.lon for p in points],
    alts=[p.alt for p in points],
    vxs=[0.0 for p in points],
    vys=[0.0 for p in points],
    vzs=[0.0 for p in points],
    cross_section_model=ConstantRcsModel(rcs=1.0),
)

# Visualization

In [ ]:
fig, ax = plt.subplots()
data = build_earth_curvature(p_start, p_stop) / 1e3
ax.plot(data[:, 0], data[:, 1], "-", label="earth surface at sea level", color="black")
data = build_earth_curvature(p_start, p_stop, 100_000) / 1e3
ax.plot(data[:, 0], data[:, 1], "--", label="Karman line ('space')", color="black")

ax.plot(
    trajectory_data[:, 0] / 1e3,
    trajectory_data[:, 1] / 1e3,
    "-",
    label=rf"Trajectory for $\alpha = {alpha} \degree$; T = {times[-1] / 60:.1f}min; $v_0$ = {v0:.0f} m/s",
)

ax.annotate("Kaliningrad", (0, 0), xytext=(0, -15))
ax.annotate("Bern", (1200, 0), xytext=(1200, -15))

ax.set_xlabel("LOS distance [km]", fontsize=14)
ax.set_ylabel("Altitude above LOS [km]", fontsize=14)
ax.legend()
ax.grid(True)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_title("Ballistic Missile Trajectory (Oreshnik-like)", fontsize=14)

fig.tight_layout()

In [ ]:
# We do not want to simulate the entire trajectory
# because that would involve too many terrain files.
LAT_MAX = 48.1311

i = next((i for i, lat in enumerate(trajectory.lats) if lat <= LAT_MAX))

trajectory_truncated = Trajectory(
    target_id=trajectory.target_id,
    target_sidc=trajectory.target_sidc,
    times=trajectory.times[i:],
    lats=trajectory.lats[i:],
    lons=trajectory.lons[i:],
    alts=trajectory.alts[i:],
    vxs=trajectory.vxs[i:],
    vys=trajectory.vys[i:],
    vzs=trajectory.vzs[i:],
    cross_section_model=trajectory.cross_section_model,
)

In [ ]:
m = folium.Map((p_stop.lat, p_stop.lon), zoom_start=4)
folium.LatLngPopup().add_to(m)
folium.GeoJson(trajectory.to_geojson(), tooltip="Trajectory full").add_to(m)
folium.GeoJson(
    trajectory_truncated.to_geojson(), color="red", tooltip="Trajectory simulated"
).add_to(m)
m

# Save to disk

In [ ]:
trajectory_truncated.times[0]

In [ ]:
from theia.simulation.scenario_import import (
    DirectFireEffectorFactory,
    FixedPathOneWayDroneFactory,
    MobileDispositive,
    TerrainFactory,
)

terrain_factory = TerrainFactory(terrain_name="SRTM")

with open("../../scenarios/ballistic_missile.json", "w") as file:
    file.write(
        MobileDispositive(
            oneway_drones=[
                FixedPathOneWayDroneFactory(
                    effector=DirectFireEffectorFactory(
                        id=0,
                        name="ballistic missile",
                        point=p_start,
                        combat_range=100.0,
                        n_attacks_left=1,
                        terrain=terrain_factory,
                    ),
                    trajectory=trajectory_truncated,
                    assigned_goal=p_stop,
                    terrain=terrain_factory,
                )
            ]
        ).model_dump_json(indent=4)
    )